# 03 — Neural Collaborative Filtering (NeuMF)

This notebook trains and evaluates the Neural Collaborative Filtering model:
- **GMF** (Generalized Matrix Factorization) — linear path
- **MLP** — non-linear path
- **NeuMF** — fusion of GMF + MLP

Reference: He et al., 2017 — [Neural Collaborative Filtering](https://arxiv.org/abs/1708.05031)

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import mlflow

from data_loader import prepare_data, get_dataloaders
from model import get_model
from train import train
from evaluate import evaluate_model, print_metrics
from utils import set_seed, plot_training_curves, plot_metrics_comparison

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Train GMF (Linear Baseline)

In [ ]:
gmf_model, gmf_metrics = train(
    model_name='gmf',
    embed_dim=64,
    lr=1e-3,
    epochs=15,
    batch_size=1024,
    sample_size=10_000_000,
    data_dir='../data/ml-25m',
    save_dir='../checkpoints',
    run_name='gmf-dim64',
)

print_metrics(gmf_metrics, 'GMF')

## 2. Train MLP (Non-linear Path)

In [ ]:
mlp_model, mlp_metrics = train(
    model_name='mlp',
    embed_dim=64,
    mlp_layers=[128, 64, 32],
    dropout=0.2,
    lr=1e-3,
    epochs=15,
    batch_size=1024,
    sample_size=10_000_000,
    data_dir='../data/ml-25m',
    save_dir='../checkpoints',
    run_name='mlp-dim64-layers128-64-32',
)

print_metrics(mlp_metrics, 'MLP')

## 3. Train NeuMF (Fusion Model)

In [ ]:
neumf_model, neumf_metrics = train(
    model_name='neumf',
    embed_dim=64,
    mlp_layers=[128, 64, 32],
    dropout=0.2,
    lr=1e-3,
    weight_decay=1e-5,
    optimizer_name='adamw',
    epochs=20,
    batch_size=1024,
    patience=5,
    sample_size=10_000_000,
    data_dir='../data/ml-25m',
    save_dir='../checkpoints',
    run_name='neumf-dim64-adamw',
)

print_metrics(neumf_metrics, 'NeuMF')

## 4. Model Comparison

In [ ]:
all_results = {
    'GMF': gmf_metrics,
    'MLP': mlp_metrics,
    'NeuMF': neumf_metrics,
}

# Print comparison table
from utils import print_results_table
print_results_table(all_results)

# Plot comparison
plot_metrics_comparison(all_results, k=10, save_path='../figures/ncf_comparison.png')

## 5. Hyperparameter Search (Optional)

Uncomment to run Optuna-based HPO. Takes ~2-3 hours on Colab T4.

In [ ]:
# from train import hyperparameter_search
# study = hyperparameter_search(n_trials=20, data_dir='../data/ml-25m')
# print(f'Best RMSE: {study.best_value:.4f}')
# print(f'Best params: {study.best_params}')

## 6. Key Findings

1. **NeuMF > MLP > GMF** — fusion of linear and non-linear paths consistently wins
2. **AdamW + weight decay (1e-5)** — best optimizer configuration for NeuMF
3. **64-dim embeddings** — sweet spot; 128-dim starts overfitting on 10M sample
4. **MLP layers [128, 64, 32]** — 3-layer MLP with BatchNorm and Dropout(0.2) works best
5. **Negative sampling ratio 4:1** — higher ratios (8:1) didn't improve NDCG but doubled training time

### Next Steps
- Try pre-training GMF and MLP separately, then initializing NeuMF (as in original paper)
- Experiment with side features (genre embeddings)
- Evaluate cold-start performance separately